# Insurance Policy Agent — Workflow with Live Steps

Run cells **top to bottom**.

**Start Jupyter from the project:**
```bash
cd ~/capstone && source .venv/bin/activate && jupyter notebook notebooks/agent_workflow.ipynb
```

**Requirements:** `.env` configured (Ollama or OpenRouter). Cell 4 prints each LangGraph step as it runs; live inference usually takes **1–5 minutes**.

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="urllib3 v2 only supports OpenSSL")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langgraph")

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Works when opened from notebooks/ or project root
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT)
load_dotenv(ROOT / ".env")

from src.config import LLM_PROVIDER, get_llm_mode_label

print("Working directory:", ROOT)
print("LLM provider:", LLM_PROVIDER)
print("Model label:", get_llm_mode_label())
print("Ready.", flush=True)

In [ ]:
# Quick LLM ping — should finish in a few seconds
from src.llm.client import complete

print("Testing one LLM call...", flush=True)
reply = complete("You are a test assistant.", "Reply with exactly: ok", temperature=0)
print("LLM replied:", repr(reply), flush=True)

## Run full workflow (streaming steps)

Each line prints when a graph node finishes: `intake` → `index` → `ingest` → ToT loop → `synthesize`.

In [ ]:
from pathlib import Path

from src.config import get_llm_mode_label
from src.graph.workflow import build_graph
from src.runner import build_user_messages


def run_agent_streaming(
    policy_paths: list[str],
    user_messages: list[str] | None = None,
    *,
    dry_run: bool = False,
    beam_width: int = 2,
    max_depth: int = 2,
    max_llm_calls: int = 25,
    start_depth: int = 1,
) -> dict:
    """Run LangGraph and print one line per completed node."""
    app = build_graph()
    state = {
        "dry_run": dry_run,
        "user_messages": user_messages
        or build_user_messages(
            age=35,
            location="Cedar Park, TX",
            flood_zone=True,
            coverage_breadth=0.4,
            low_cost=0.3,
            few_exclusions=0.3,
        ),
        "policy_paths": policy_paths,
        "llm_call_count": 0,
        "max_llm_calls": max_llm_calls,
        "beam_width": beam_width,
        "max_depth": max_depth,
        "start_depth": start_depth,
        "retrieval_cache": {},
    }

    result = dict(state)
    print(f"Mode: {'dry-run' if dry_run else get_llm_mode_label()}", flush=True)
    print(f"Policies: {[Path(p).name for p in policy_paths]}", flush=True)
    print(f"beam_width={beam_width} max_depth={max_depth}\n", flush=True)

    for step in app.stream(state, config={"recursion_limit": 50}, stream_mode="updates"):
        for node_name, update in step.items():
            if "llm_call_count" in update:
                result["llm_call_count"] = update["llm_call_count"]
            result.update(update)
            calls = result.get("llm_call_count", 0)
            depth = result.get("depth")
            extra = f"depth={depth}" if depth is not None and node_name in ("tot_expand", "evaluate", "prune_beam", "advance_depth") else ""
            branches = update.get("active_branches")
            if branches is not None and node_name in ("tot_expand", "evaluate", "prune_beam"):
                extra += f" branches={len(branches)}"
            if "indexed_chunks" in update:
                extra += f" chunks={update['indexed_chunks']}"
            if "normalized_plans" in update:
                extra += f" plans={len(update['normalized_plans'])}"
            if update.get("error"):
                extra += f" ERROR={update['error']}"
            suffix = f" ({extra.strip()})" if extra.strip() else ""
            print(f"  ✓ {node_name}{suffix}  [LLM calls: {calls}]", flush=True)

    result["mode"] = "dry_run" if dry_run else get_llm_mode_label()
    print("\n--- DONE ---", flush=True)
    print("Mode:", result.get("mode"))
    print("LLM calls:", result.get("llm_call_count"))
    if result.get("error"):
        print("Error:", result["error"])
    return result


policies = [
    str(ROOT / "data/synthetic/plan_a.txt"),
    str(ROOT / "data/synthetic/plan_b.txt"),
]
messages = build_user_messages(
    age=35,
    location="Cedar Park, TX",
    flood_zone=True,
    coverage_breadth=0.4,
    low_cost=0.3,
    few_exclusions=0.3,
)

result = run_agent_streaming(
    policy_paths=policies,
    user_messages=messages,
    beam_width=2,
    max_depth=2,
)

## Inspect workflow state

Profile from intake, normalized plans, branch scores from Tree-of-Thought, and the winning branch.

In [ ]:
import json
from pprint import pprint


def inspect_workflow(state: dict) -> None:
    print("=" * 60)
    print("SESSION PROFILE (intake agent)")
    print("=" * 60)
    pprint(state.get("session_profile", {}))

    print("\n" + "=" * 60)
    print("INDEX & PLANS")
    print("=" * 60)
    print("Indexed chunks:", state.get("indexed_chunks"))
    for plan in state.get("normalized_plans", []):
        pid = plan.get("plan_id", "?")
        carrier = plan.get("carrier", "unknown")
        premium = plan.get("annual_premium")
        print(f"  · {pid}: carrier={carrier}, premium={premium}")

    branches = state.get("active_branches") or []
    if state.get("winning_branch"):
        branches = [state["winning_branch"]] + [
            b for b in branches if b.get("branch_id") != state["winning_branch"].get("branch_id")
        ]

    print("\n" + "=" * 60)
    print("BRANCHES (Tree-of-Thought)")
    print("=" * 60)
    if not branches:
        print("  (no branches in final state)")
    for b in branches:
        winner = state.get("winning_branch", {}).get("branch_id") == b.get("branch_id")
        tag = " ★ WINNER" if winner else ""
        pruned = " [PRUNED]" if b.get("pruned") else ""
        hard = " [HARD GATE FAIL]" if b.get("hard_gate_failed") else ""
        print(
            f"\n  {b.get('branch_id')}{tag}{pruned}{hard}"
            f"  depth={b.get('depth')}  score={b.get('composite_score', 0):.3f}"
        )
        if b.get("interpretation"):
            print(f"    interpretation: {b['interpretation'][:120]}")
        if b.get("rationale"):
            print(f"    rationale: {b['rationale'][:160]}")
        scores = b.get("scores") or {}
        if scores:
            print("    scores:", json.dumps(scores, indent=6))
        for i, thought in enumerate(b.get("thoughts") or [], 1):
            claim = (thought.get("claim") or "")[:80]
            payout = thought.get("payout")
            print(
                f"    thought {i}: plan={thought.get('plan_id')} "
                f"scenario={thought.get('scenario_id')} payout={payout} — {claim}"
            )

    wb = state.get("winning_branch")
    if wb:
        print("\n" + "=" * 60)
        print("WINNING BRANCH SUMMARY")
        print("=" * 60)
        print(f"  id: {wb.get('branch_id')}")
        print(f"  composite_score: {wb.get('composite_score')}")
        print(f"  rationale: {wb.get('rationale', '')[:200]}")


inspect_workflow(result)

## Final recommendation

In [ ]:
from IPython.display import Markdown, display

display(Markdown(result.get("final_recommendation") or "_No recommendation generated._"))

## Optional: public HO-3 specimen policies

Re-run with real regulatory specimen forms (Travelers HO-3 + State Farm).

In [ ]:
public_policies = [
    str(ROOT / "data/public/travelers_ho3_nv.txt"),
    str(ROOT / "data/public/statefarm_hw2136_ok.txt"),
]

result_public = run_agent_streaming(
    policy_paths=public_policies,
    user_messages=messages,
    beam_width=2,
    max_depth=2,
)

inspect_workflow(result_public)
display(Markdown(result_public.get("final_recommendation") or "_No output._"))